In [8]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/census_2001/PC01_C01_27.xls")

# Display all worksheet names
excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)

# Inspect the first 20 rows without assuming a header
df_preview = pd.read_excel(
    file_path,
    sheet_name=excel_file.sheet_names[0],
    header=None,
    nrows=20
)

print(df_preview.to_string(index=False, header=False))

['C01T']
  NaN   NaN    NaN    NaN      NaN         NaN    NaN C -1 POPULATION BY RELIGIOUS COMMUNITY      NaN      NaN                    NaN      NaN      NaN      NaN     NaN     NaN       NaN    NaN     NaN C -1 POPULATION BY RELIGIOUS COMMUNITY    NaN     NaN      NaN     NaN     NaN     NaN    NaN     NaN                             NaN    NaN     NaN                 NaN   NaN     NaN
Table State Distt. Tehsil     Town   Area Name Total/                                    NaN      NaN      NaN  Religious communities      NaN      NaN      NaN     NaN     NaN       NaN    NaN     NaN                                    NaN    NaN     NaN      NaN     NaN     NaN     NaN    NaN     NaN                             NaN    NaN     NaN                 NaN   NaN     NaN
 Name  Code   Code   Code     Code         NaN Rural/                                  Total      NaN      NaN                  Hindu      NaN      NaN   Muslim     NaN     NaN Christian    NaN     NaN                    

In [9]:

import pandas as pd
from pathlib import Path

# 1. Define file paths
input_file = Path("../data/raw/census_2001/PC01_C01_27.xls")
output_file = Path("../data/processed/census_2001_cleaned.csv")

# Create output folder if it doesn't exist
output_file.parent.mkdir(parents=True, exist_ok=True)

# 2. Load the Excel file without assuming column headers
df = pd.read_excel(
    input_file,
    sheet_name="C01T",
    header=None
)

print("Original dataset shape:", df.shape)

# 3. Remove the first 6 rows containing report headings
df = df.iloc[6:].reset_index(drop=True)

# 4. Assign meaningful column names
columns = [
    "Table_Code",
    "State_Code",
    "District_Code",
    "Tehsil_Code",
    "Town_Code",
    "Area_Name",
    "Area_Type",
    "Total_Persons",
    "Total_Males",
    "Total_Females",
    "Hindu_Persons",
    "Hindu_Males",
    "Hindu_Females",
    "Muslim_Persons",
    "Muslim_Males",
    "Muslim_Females",
    "Christian_Persons",
    "Christian_Males",
    "Christian_Females",
    "Sikh_Persons",
    "Sikh_Males",
    "Sikh_Females",
    "Buddhist_Persons",
    "Buddhist_Males",
    "Buddhist_Females",
    "Jain_Persons",
    "Jain_Males",
    "Jain_Females",
    "Other_Religions_Persons",
    "Other_Religions_Males",
    "Other_Religions_Females",
    "Religion_Not_Stated_Persons",
    "Religion_Not_Stated_Males",
    "Religion_Not_Stated_Females"
]

# 5. Remove the remaining header/separator rows
df = df.iloc[1:].reset_index(drop=True)

# 6. Apply column names only if the column count matches
if df.shape[1] != len(columns):
    raise ValueError(
        f"Column count mismatch: found {df.shape[1]} columns, "
        f"but expected {len(columns)}. Inspect the source file."
    )

df.columns = columns

# 7. Remove completely empty rows
df = df.dropna(how="all").copy()

# 8. Clean text columns
text_columns = [
    "Table_Code",
    "Area_Name",
    "Area_Type"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

# 9. Remove rows without a valid area name or population record
df = df[
    df["Area_Name"].notna()
    & df["Total_Persons"].notna()
].copy()

# 10. Convert population columns to numeric values
population_columns = columns[7:]

for col in population_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 11. Save cleaned data
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Cleaned dataset shape:", df.shape)
print("Saved to:", output_file)

# 12. Preview cleaned data
print(df.head(10).to_string(index=False))



Original dataset shape: (1554, 34)
Cleaned dataset shape: (1547, 34)
Saved to: ..\data\processed\census_2001_cleaned.csv
Table_Code State_Code District_Code Tehsil_Code Town_Code   Area_Name Area_Type  Total_Persons  Total_Males  Total_Females  Hindu_Persons  Hindu_Males  Hindu_Females  Muslim_Persons  Muslim_Males  Muslim_Females  Christian_Persons  Christian_Males  Christian_Females  Sikh_Persons  Sikh_Males  Sikh_Females  Buddhist_Persons  Buddhist_Males  Buddhist_Females  Jain_Persons  Jain_Males  Jain_Females  Other_Religions_Persons  Other_Religions_Males  Other_Religions_Females  Religion_Not_Stated_Persons  Religion_Not_Stated_Males  Religion_Not_Stated_Females
     C0101         27            00        0000  00000000 MAHARASHTRA     Total       96878627     50400596       46478031       77859385     40496085       37363300        10270485       5437926         4832559            1058313           530975             527338        215337      117736         97601           58387

In [10]:
input_file = Path("../data/raw/census_2001/PC01_C01_27.xls")

In [11]:

import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/census_2001_cleaned.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nArea types:")
print(df["Area_Type"].value_counts(dropna=False))

print("\nFirst 10 records:")
print(df[["Area_Name", "Area_Type", "Total_Persons",
          "Hindu_Persons", "Muslim_Persons"]].head(10))

print("\nMissing values:")
print(df.isnull().sum())

print("\nMaharashtra state totals:")
print(
    df[df["Area_Name"].str.upper().str.strip() == "MAHARASHTRA"]
    [["Area_Name", "Area_Type", "Total_Persons",
      "Hindu_Persons", "Muslim_Persons"]]
)



Dataset shape: (1547, 34)

Column names:
['Table_Code', 'State_Code', 'District_Code', 'Tehsil_Code', 'Town_Code', 'Area_Name', 'Area_Type', 'Total_Persons', 'Total_Males', 'Total_Females', 'Hindu_Persons', 'Hindu_Males', 'Hindu_Females', 'Muslim_Persons', 'Muslim_Males', 'Muslim_Females', 'Christian_Persons', 'Christian_Males', 'Christian_Females', 'Sikh_Persons', 'Sikh_Males', 'Sikh_Females', 'Buddhist_Persons', 'Buddhist_Males', 'Buddhist_Females', 'Jain_Persons', 'Jain_Males', 'Jain_Females', 'Other_Religions_Persons', 'Other_Religions_Males', 'Other_Religions_Females', 'Religion_Not_Stated_Persons', 'Religion_Not_Stated_Males', 'Religion_Not_Stated_Females']

Area types:
Area_Type
Urban    769
Total    389
Rural    389
Name: count, dtype: int64

First 10 records:
     Area_Name Area_Type  Total_Persons  Hindu_Persons  Muslim_Persons
0  MAHARASHTRA     Total       96878627       77859385        10270485
1  MAHARASHTRA     Rural       55777647       48786036         3081552
2  MAHAR

In [12]:

import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/census_2001_cleaned.csv")

df = pd.read_csv(file_path)

# Display all Maharashtra rows
maharashtra = df[
    df["Area_Name"].str.strip().str.upper() == "MAHARASHTRA"
]

print(maharashtra[["Area_Name", "Area_Type", "Total_Persons"]])

# Verify the rural + urban population
rural = maharashtra.loc[
    maharashtra["Area_Type"] == "Rural", "Total_Persons"
].sum()

urban = maharashtra.loc[
    maharashtra["Area_Type"] == "Urban", "Total_Persons"
].sum()

print("\nRural population:", rural)
print("Urban population:", urban)
print("Rural + Urban:", rural + urban)
print("Expected state population:", 96878627)
print("Difference:", 96878627 - (rural + urban))



     Area_Name Area_Type  Total_Persons
0  MAHARASHTRA     Total       96878627
1  MAHARASHTRA     Rural       55777647
2  MAHARASHTRA     Urban       41100980

Rural population: 55777647
Urban population: 41100980
Rural + Urban: 96878627
Expected state population: 96878627
Difference: 0


In [13]:

# Inspect the original Excel file
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/census_2001/PC01_C01_27.xls")

raw_df = pd.read_excel(
    file_path,
    sheet_name="C01T",
    header=None
)

# Display the rows containing Maharashtra's state-level records
print(raw_df.iloc[6:12].to_string(index=False, header=False))



  NaN NaN NaN  NaN      NaN         NaN   NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN     NaN     NaN     NaN    NaN    NaN    NaN    NaN   NaN     NaN     NaN     NaN     NaN    NaN    NaN    NaN    NaN    NaN   NaN   NaN   NaN
C0101  27  00 0000 00000000 MAHARASHTRA Total 96878627 50400596 46478031 77859385 40496085 37363300 10270485 5437926 4832559 1058313 530975 527338 215337 117736 97601 5838710 2977607 2861103 1301843 670236 631607 236841 119113 117728 97713 50918 46795
C0101  27  00 0000 00000000 MAHARASHTRA Rural 55777647 28458677 27318970 48786036 24890550 23895486  3081552 1577482 1504070  159577  80266  79311  29611  16772 12839 3228573 1639513 1589060  286557 149467 137090 132766  67015  65751 72975 37612 35363
C0101  27  00 0000 00000000 MAHARASHTRA Urban 41100980 21941919 19159061 29073349 15605535 13467814  7188933 3860444 3328489  898736 450709 448027 185726 100964 84762 2610137 1338094 1272043 1015286 520769 494517 104075  52098  51977 24738 1330